In [ ]:
import pandas as pd
import numpy as np
from typing import List
from sklearn.metrics import roc_auc_score
import xgboost as xgb


# =========================================================
# 🔧 FEATURE ENGINEERING AUTOMATIQUE
# =========================================================

def create_time_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df["day"] = df[date_col].dt.day
    df["month"] = df[date_col].dt.month
    df["dayofweek"] = df[date_col].dt.dayofweek
    df["dayofyear"] = df[date_col].dt.dayofyear
    df["weekofyear"] = df[date_col].dt.isocalendar().week.astype(int)
    return df


def create_lag_features(df: pd.DataFrame, cols: List[str], lags: List[int]) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        for lag in lags:
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)
    return df


def create_rolling_features(df: pd.DataFrame, cols: List[str], windows: List[int]) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        for window in windows:
            df[f"{col}_roll_mean_{window}"] = df[col].rolling(window).mean()
            df[f"{col}_roll_std_{window}"] = df[col].rolling(window).std()
    return df


def build_features(df: pd.DataFrame, date_col: str, target_col: str) -> pd.DataFrame:
    df = df.sort_values(date_col).copy()

    # --- Features temporelles
    df = create_time_features(df, date_col)

    # --- Lags
    df = create_lag_features(df, cols=[target_col, "temp", "humidity"], lags=[1, 2, 3, 7])

    # --- Rolling stats
    df = create_rolling_features(df, cols=[target_col, "temp"], windows=[3, 7])

    return df


# =========================================================
# 🎯 PIPELINE XGBOOST
# =========================================================

def train_xgboost_pipeline(df: pd.DataFrame):
    
    df = df.copy()
    
    # --- Création target (pluie demain)
    df["target"] = df["rain"].shift(-1) > 0

    # --- Features
    df = build_features(df, date_col="date", target_col="rain")

    # --- Clean
    df = df.dropna()

    # --- Split temporel
    split = int(len(df) * 0.8)

    X = df.drop(columns=["date", "target"])
    y = df["target"].astype(int)

    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]

    # --- Modèle XGBoost
    model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # --- Evaluation
    y_pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred)

    print(f"AUC: {auc:.4f}")

    return model, X.columns


# =========================================================
# 🚀 EXECUTION
# =========================================================

if __name__ == "__main__":
    df = pd.read_csv("weather.csv", parse_dates=["date"])
    
    model, features = train_xgboost_pipeline(df)

    print("\nFeatures utilisées :")
    print(features.tolist())